[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/evals/trace_level_evals.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://docs.arize.com/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-1px8dcmlf-fmThhDFD_V_48oU7ALan4Q">社区</a>
    </p>
</center>


# 追踪级评估：超越输入输出检查

*完整示例：电影推荐 Agent。*

## 为什么输入输出检查还不够

评估 LLM 应用最常见的方式是**端到端**评估：获取用户请求和最终答案，判断答案好不好。这适用于单次模型调用，却不适用于 **Agent**，因为 Agent 是一系列*决策*：调用哪些工具、按什么顺序调用、使用哪些参数，以及如何根据结果进行推理。

端到端评估只检查输入和最终输出这两个**端点**。值得关注的失败发生在看不到的中间过程：

- **推理**：Agent 是否围绕目标进行连贯推理，还是碰巧得到答案？
- **工具选择**：是否选择了正确工具，并*跳过*错误工具？
- **决策路径**：是否按合理顺序、使用合理参数调用工具？

只看端点会在两个方向上误导你。**看似正确的答案可能来自有问题的路径**：错误的理由碰巧产生正确结果，输入稍一变化就会失败。而答案确实错误时，端点只能告诉你*错了*，无法解释*为什么*；只有查看追踪内部才能看到根因。

**方法：**捕获完整追踪，根据 span 重建 Agent 的*决策路径*，评估中间步骤，而不只是 `input → output`。

我们用电影推荐 Agent 演示这些思路。本 Notebook 将带你：

- 构建电影推荐 Agent 并捕获其交互追踪
- 从 span 中重建 Agent 的**决策路径**，即有序工具调用及其参数
- 运行**三个评估器**：一个*端点*检查（推荐相关性）和两个*中间过程*检查，即决策路径（工具选择、顺序和参数）及推理（答案是否以工具结果为依据）
- 观察各评估器如何发现端点检查遗漏的失败
- 将评估输出整理为 Phoenix 要求的格式并写回平台

✅ 运行本 Notebook 需要免费的 [Phoenix Cloud 账号](https://app.arize.com/auth/phoenix/login) 和 OpenAI API 密钥。

# 配置密钥与依赖


In [ ]:
%pip install openinference-instrumentation-openai openinference-instrumentation-openai-agents openinference-instrumentation arize-phoenix arize-phoenix-otel nest_asyncio openai openai-agents

In [ ]:
import os
from getpass import getpass

if not (phoenix_endpoint := os.getenv("PHOENIX_COLLECTOR_ENDPOINT")):
    phoenix_endpoint = getpass("🔑 请输入 Phoenix 收集器端点：")
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = phoenix_endpoint


if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
    phoenix_api_key = getpass("🔑 请输入 Phoenix API 密钥：")
os.environ["PHOENIX_API_KEY"] = phoenix_api_key

if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥：")
os.environ["OPENAI_API_KEY"] = openai_api_key

# 配置追踪


In [ ]:
from phoenix.otel import register

# 配置 Phoenix 追踪器
tracer_provider = register(project_name="movie-rec-agent", auto_instrument=True)

首先定义推荐系统使用的工具。本例定义三个工具：

1. 电影选择器：根据用户指定的类型，选择最多 5 部近期可在线播放的电影。
2. 评论工具：查找电影评论；如果输入电影列表，则按评分从高到低排序。
3. 简介摘要工具：为每部电影返回 1～2 句简介。

理想流程是用户只需提供想看的电影类型，系统就返回包含简介和评论的候选列表。

Agent 的每次工具调用都是一个**决策点**，并作为 span 记录在追踪中。这一串 span 就是之后要重建和评估的决策路径，也是端到端评估无法看到的部分。

测试 Agent，并在 Phoenix 中查看追踪。

In [ ]:
import ast
from typing import List, Union

from agents import Agent, Runner, function_tool
from openai import OpenAI
from opentelemetry import trace

tracer = trace.get_tracer(__name__)

client = OpenAI()


@function_tool
def movie_selector_llm(genre: str) -> List[str]:
    prompt = (
        f"列出最多 5 部近期热门、可在线播放的 {genre} 类型电影。"
        "仅以 Python 字符串列表提供片名。"
    )
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=150,
    )
    content = response.choices[0].message.content
    try:
        movie_list = ast.literal_eval(content)
        if isinstance(movie_list, list):
            return movie_list[:5]
    except Exception:
        return content.split("\n")


@function_tool
def reviewer_llm(movies: Union[str, List[str]]) -> str:
    if isinstance(movies, list):
        movies_str = ", ".join(movies)
        prompt = f"将以下电影按评分从高到低排序，并分别提供简短评论：\n{movies_str}"
    else:
        prompt = f"为以下电影提供简短评论和评分：{movies}"
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=300,
    )
    return response.choices[0].message.content.strip()


@function_tool
def preview_summarizer_llm(movie: str) -> str:
    prompt = f"用 1～2 句话概述电影 '{movie}'。"
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=100,
    )
    return response.choices[0].message.content.strip()

In [ ]:
agent = Agent(
    name="MovieRecommendationAgentLLM",
    tools=[movie_selector_llm, reviewer_llm, preview_summarizer_llm],
    instructions=(
        "你是乐于助人的电影推荐助手，可以使用三个工具：\n"
        "1. MovieSelector：给定类型，返回最多 5 部近期可在线播放的电影。\n"
        "2. Reviewer：给定一部或多部电影，返回评论并按评分排序。\n"
        "3. PreviewSummarizer：给定片名，返回 1～2 句摘要。\n\n"
        "你的目标是结合相关信息，给出有帮助且易于理解的回复。"
    ),
)


async def main():
    user_input = "Which comedy movie should I watch?"
    result = await Runner.run(agent, user_input)
    print(result.final_output)


await main()

接下来多运行几次 Agent，生成更多追踪。你可以按需调整或自定义问题。


In [ ]:
import time

questions = [
    "Which Batman movie should I watch?",
    "I want to watch a good romcom",
    "What is a very scary horror movie?",
    "Name a feel-good holiday movie",
    "Recommend a musical with great songs",
    "Give me a classic drama from the 90s",
]

for question in questions:
    result = await Runner.run(agent, question)

# span 在后台导出，因此先强制刷新，并给 Phoenix
# 一点时间接收，再在下一节查询。
tracer_provider.force_flush()
time.sleep(5)

# 从 Phoenix 获取 span 数据


运行评估前，从 Phoenix 检索 span 数据，并整理各评估器所需的内容。

先提取**端点**：用户问题（`input`）和 Agent 的*最终答案*（`output`）。只获取最终答案，不拼接所有 span 的输出。混入工具输出会模糊“用户看到的内容”与“追踪内部发生的过程”之间的界线，削弱本教程对端点评估与追踪评估的比较。

> **为什么读取 LLM span，而不是根 span？** 使用 OpenAI Agents 插桩时，根 `AGENT` span 不记录 `input.value` / `output.value`；这些属性位于底层 `LLM` span。因此使用 OpenInference 标准属性 `llm.input_messages` / `llm.output_messages`，从第一次 LLM 调用中读取用户问题，从最后一个文本轮次中读取最终回复。如果插桩确实填充了根 span（很多插桩如此），则可直接从 `agent_roots` 读取 `input.value` / `output.value`。

端到端评估只能看到这些端点。下一步重建它遗漏的中间信号。

In [ ]:
from phoenix.client import AsyncClient

px_client = AsyncClient()
primary_df = await px_client.spans.get_spans_dataframe(project_identifier="movie-rec-agent")

In [ ]:
import json

import pandas as pd


def _as_list(value):
    """span 属性可能是 JSON 字符串，也可能是已解析列表。"""
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return []
    return value if isinstance(value, list) else []


def user_query(input_messages):
    """LLM span 输入中的第一条用户消息。"""
    for message in _as_list(input_messages):
        if message.get("message.role") == "user":
            contents = message.get("message.contents") or []
            return message.get("message.content") or "".join(
                c.get("message_content.text", "") for c in contents
            )
    return None


def assistant_text(output_messages):
    """LLM span 中的助手文本回复，工具调用轮次为空。"""
    texts = []
    for message in _as_list(output_messages):
        if message.get("message.role") == "assistant":
            if message.get("message.content"):
                texts.append(message["message.content"])
            for c in message.get("message.contents") or []:
                if c.get("message_content.type") == "text":
                    texts.append(c.get("message_content.text", ""))
    return " ".join(t for t in texts if t).strip() or None


# 仅保留 Agent 自身的追踪：每条都有一个无父节点的根 span，其类型为
# AGENT。这样可排除评估器自身的 LLM 调用；运行后续单元后，这些调用
# 也会被追踪到此项目，但不应纳入评估集。
agent_roots = primary_df[(primary_df["parent_id"].isna()) & (primary_df["span_kind"] == "AGENT")]
agent_trace_ids = agent_roots["context.trace_id"].unique()

llm_spans = primary_df[
    (primary_df["span_kind"] == "LLM") & (primary_df["context.trace_id"].isin(agent_trace_ids))
].sort_values("start_time")

# 端点输入 = 用户问题，即追踪中的第一条用户消息。
# 端点输出 = 仅 Agent 的最终答案，即最后一个文本轮次；不要
# 拼接每个 span 的输出，否则会把中间工具结果泄露到
# 本应用于表示“用户看到的内容”的数据中。
trace_df = pd.DataFrame(
    {
        "input": llm_spans.groupby("context.trace_id")["attributes.llm.input_messages"].apply(
            lambda s: next((q for q in s.map(user_query) if q), None)
        ),
        "output": llm_spans.groupby("context.trace_id")["attributes.llm.output_messages"].apply(
            lambda s: next((a for a in reversed(list(s.map(assistant_text))) if a), None)
        ),
    }
)
# 丢弃无法恢复两个端点的追踪，避免评估器处理
# 空输入或空输出。
trace_df = trace_df.dropna(subset=["input", "output"])
trace_df.head()

## 重建中间信号

上面的 `output` 是端点，即 Agent 的最终答案。为了评估 Agent 的*过程*，我们从追踪 span 中重建两个端点无法展示的中间信号（筛选 `span_kind == "TOOL"`，按 `start_time` 排序）：

- **`tool_path`**：Agent 按顺序进行的工具调用，*包含参数*。这就是决策路径：工具选择、顺序以及各次调用的参数。
- **`tool_io`**：每个工具*接收了什么*、*返回了什么*。它帮助判断最终答案是否真正以工具结果为依据，而非编造出貌似可信的内容。

这两列正是端到端评估舍弃的信号。

In [ ]:
tool_spans = primary_df[primary_df["span_kind"] == "TOOL"].sort_values("start_time")


# tool_path：包含参数的有序工具调用，即决策路径，
# 记录工具选择、顺序，以及每个工具被要求做什么。
def format_decision_path(group):
    return " -> ".join(
        f"{row['name']}({row['attributes.input.value']})" for _, row in group.iterrows()
    )


trace_df["tool_path"] = (
    tool_spans.groupby("context.trace_id")[["name", "attributes.input.value"]]
    .apply(format_decision_path)
    .reindex(trace_df.index)
    .fillna("No tools called")
)


# tool_io：每次工具调用的输入和输出，使评估器能够检查
# 最终答案是否以工具实际返回的内容为依据。
def format_tool_calls(group):
    lines = []
    for i, (_, row) in enumerate(group.iterrows(), start=1):
        lines.append(
            f"{i}. {row['name']} | input: {row['attributes.input.value']} "
            f"| output: {row['attributes.output.value']}"
        )
    return "\n".join(lines)


trace_df["tool_io"] = (
    tool_spans.groupby("context.trace_id")[
        ["name", "attributes.input.value", "attributes.output.value"]
    ]
    .apply(format_tool_calls)
    .reindex(trace_df.index)
    .fillna("No tools called")
)
trace_df[["input", "tool_path", "tool_io"]].head()

# 定义并运行评估器


现在运行**三个评估器**：一个端点检查和两个中间过程检查，各自读取不同列：

1. **推荐相关性：*端点*检查。**读取 `input` 和最终 `output`，判断推荐是否符合请求。这正是端到端评估所做的事。
2. **决策路径：*中间过程*检查。**读取 `input` 和 `tool_path`（有序工具调用及其参数），判断 Agent 是否选择正确工具、顺序是否合理、参数是否恰当。它不查看最终输出，只评判 Agent *如何*得到答案。
3. **推理与依据：*中间过程*检查。**读取 `input`、`tool_io` 和 `output`，判断最终答案是否真正*以工具结果为依据*，发现任何工具都未生成的评分、评论或片名等事实。

本 Notebook 的关键是这种对比：追踪可以**通过端点检查，却未通过中间过程检查**，例如在工具顺序错误的情况下得出好看的答案，或编造工具从未返回的事实。三者都使用 LLM 裁判分类器。

In [ ]:
DECISION_PATH = """
你正在评估 Agent 的决策路径：为回答请求所执行的有序工具调用，
包括哪些工具、什么顺序以及哪些参数。你不评判最终答案文本，
也不判断各工具输出是否正确，只判断 Agent 的选择是否合理。

Agent 有三个可用工具：
- movie_selector_llm(genre)：返回候选电影。必须最先调用，
  因为其他工具操作的是已选择的电影。
- reviewer_llm(movies)：评论并排序电影。只能在选择之后调用，
  且应传入实际选定的电影。
- preview_summarizer_llm(movie)：总结电影。只能在选择之后调用。

你将收到：
1. 发起追踪的用户输入
2. Agent 执行的有序工具调用及其参数

##
用户输入：
{input}

决策路径（有序工具调用及参数）：
{tool_path}
##

必须只回复一个词：`correct` 或 `incorrect`。
1. `correct` →
- 在 reviewer_llm 或 preview_summarizer_llm 之前调用 movie_selector_llm，并且
- 每个工具都使用合理参数，例如 reviewer_llm 接收的是已选定电影，
  而非空列表或无关列表。
2. `incorrect` →
- 操作电影的工具 reviewer_llm / preview_summarizer_llm 在选定电影之前运行，
  或缺少选择步骤，或工具参数不合理，
  例如空输入、占位输入或从未选定的电影。
"""

In [ ]:
RECOMMENDATION_RELEVANCE = """
你正在评估 LLM 应用提供的电影推荐相关性。

你将收到：
1. 发起追踪的用户输入
2. 系统输出的电影推荐列表

##
用户输入：
{input}

推荐：
{output}
##

必须只回复一个词：`correct` 或 `incorrect`。
1. `correct` →
- 所有推荐电影符合用户输入指定的类型或条件。
- 推荐与用户请求相关且不重复。
2. `incorrect` →
- 一项或多项推荐不符合指定类型或条件，或推荐重复。
"""

In [ ]:
REASONING_SUPPORT = """
你正在检查 Agent 的最终答案是否得到工具实际返回结果的支持。

你不判断工具顺序，那属于决策路径检查；也不判断类型是否匹配，那属于相关性检查。
只判断最终答案中的每个具体断言，包括片名、评分、分数、评论引述和剧情事实，
是否以下面的工具输出为依据。如果答案声称了工具从未提供的事实，
例如某个具体评分，即使听起来合理，也属于无依据。

##
用户输入：
{input}

工具调用及结果（按顺序）：
{tool_io}

最终答案：
{output}
##

必须只回复一个词：`correct` 或 `incorrect`。
1. `correct` → 最终答案中的每个具体断言都有上述工具结果支持。
2. `incorrect` → 最终答案至少声称了一个工具结果中不存在的具体事实，
   例如评分、分数、评论或片名。
"""

In [ ]:
from phoenix.evals import LLM, ClassificationEvaluator, async_evaluate_dataframe

from phoenix.trace import suppress_tracing

llm = LLM(provider="openai", model="gpt-4o-mini")


# 端点检查：仅输入 → 最终答案，即端到端评估可见的内容。
relevance_evaluator = ClassificationEvaluator(
    name="relevance",
    llm=llm,
    prompt_template=RECOMMENDATION_RELEVANCE,
    choices={"correct": 1.0, "incorrect": 0.0},
)

# 中间过程检查：Agent 是否按正确顺序选择正确工具？
path_evaluator = ClassificationEvaluator(
    name="decision path",
    llm=llm,
    prompt_template=DECISION_PATH,
    choices={"correct": 1.0, "incorrect": 0.0},
)

# 中间过程检查：最终答案是否真正以工具结果为依据？
reasoning_evaluator = ClassificationEvaluator(
    name="reasoning",
    llm=llm,
    prompt_template=REASONING_SUPPORT,
    choices={"correct": 1.0, "incorrect": 0.0},
)


# 裁判本身也是 LLM 调用，自动插桩会将其追踪到
# 同一项目。suppress_tracing 排除这些评估器 span，
# 使项目只保留 Agent 追踪。
with suppress_tracing():
    results_df = await async_evaluate_dataframe(
        dataframe=trace_df,
        evaluators=[relevance_evaluator, path_evaluator, reasoning_evaluator],
    )
results_df.head()

## 看看各评估器能发现什么

在正常追踪中，三个评估器结论一致。追踪级评估的价值在于它们*不一致*的时候。下面使用三个裁判评估三个受控案例，它们**与真实追踪结构一致**（`input`、`output`、`tool_path`、`tool_io`），且答案看起来都相关，因此**端点相关性检查每次都通过**：

- **正常运行**：先选择，再评论，最后总结；答案与工具结果一致。全部通过。
- **顺序错误**：尚未选定电影就先评论。**决策路径**检查能发现问题，相关性与依据检查不能。
- **无依据的断言**：工具顺序正确，但答案引用了任何工具输出都未出现的 *“9.2/10”* 评分。**推理与依据**检查能发现问题，相关性和决策路径检查不能。

两种中间过程失败，需要两种不同视角；它们对端点检查都不可见。

In [ ]:
# 与真实追踪结构一致的受控案例，包含 input、output、tool_path、
# tool_io。各答案看似相关，所以端点检查通过；
# 失败存在于中间信号中。tool_path 展示调用和参数；
# tool_io 额外提供结果。每份记录都符合其路径，包括顺序错误案例。
clean_path = (
    'movie_selector_llm({"genre":"comedy"}) -> '
    'reviewer_llm({"movies":["The Grand Budapest Hotel","Superbad"]}) -> '
    'preview_summarizer_llm({"movie":"Superbad"})'
)
clean_io = (
    '1. movie_selector_llm | input: {"genre":"comedy"} | output: ["The Grand Budapest Hotel", "Superbad"]\n'
    '2. reviewer_llm | input: {"movies":["The Grand Budapest Hotel","Superbad"]} | output: The Grand Budapest Hotel — witty and stylish; Superbad — raunchy coming-of-age comedy.\n'
    '3. preview_summarizer_llm | input: {"movie":"Superbad"} | output: Two friends navigate one wild night before graduation.'
)
broken_order_path = 'reviewer_llm({"movies":[]}) -> movie_selector_llm({"genre":"comedy"})'
broken_order_io = (
    '1. reviewer_llm | input: {"movies":[]} | output: No movies were provided to review.\n'
    '2. movie_selector_llm | input: {"genre":"comedy"} | output: ["The Grand Budapest Hotel", "Superbad"]'
)
unsupported_path = (
    'movie_selector_llm({"genre":"comedy"}) -> '
    'reviewer_llm({"movies":["Superbad"]}) -> '
    'preview_summarizer_llm({"movie":"Superbad"})'
)
no_rating_io = (
    '1. movie_selector_llm | input: {"genre":"comedy"} | output: ["The Grand Budapest Hotel", "Superbad"]\n'
    '2. reviewer_llm | input: {"movies":["Superbad"]} | output: Superbad — a funny, heartfelt coming-of-age comedy.\n'
    '3. preview_summarizer_llm | input: {"movie":"Superbad"} | output: Two friends navigate one wild night.'
)

demo_df = pd.DataFrame(
    {
        "case": ["clean run", "broken order", "unsupported claim"],
        "input": ["Which comedy movie should I watch?"] * 3,
        "output": [
            "Two great comedy picks: 'The Grand Budapest Hotel' and 'Superbad'.",
            "Two great comedy picks: 'The Grand Budapest Hotel' and 'Superbad'.",
            "Watch 'Superbad' — critics rated it 9.2/10.",  # 没有工具生成过此评分
        ],
        "tool_path": [clean_path, broken_order_path, unsupported_path],
        "tool_io": [clean_io, broken_order_io, no_rating_io],
    }
)

with suppress_tracing():
    demo_results = await async_evaluate_dataframe(
        dataframe=demo_df,
        evaluators=[relevance_evaluator, path_evaluator, reasoning_evaluator],
    )


def label_of(score):
    """从 Phoenix 评分单元中提取分类标签。"""
    if isinstance(score, str):
        score = json.loads(score)
    if isinstance(score, dict):
        return score.get("label")
    return getattr(score, "label", None)


pd.DataFrame(
    {
        "case": demo_df["case"],
        "relevance": demo_results["relevance_score"].apply(label_of),
        "decision path": demo_results["decision path_score"].apply(label_of),
        "reasoning": demo_results["reasoning_score"].apply(label_of),
    }
)

# 将结果写回 Phoenix


最后将结果记录回 Phoenix。运行以下单元后，即可在平台查看追踪级评估，包括相关标签、分数和解释。

In [ ]:
from phoenix.evals.utils import to_annotation_dataframe

# 将各 trace 映射到根 AGENT span，使评估关联根 span。
root_spans = agent_roots[["context.trace_id", "context.span_id"]]

# 将结果与根 span 合并，按 trace_id 对齐
results_with_spans = pd.merge(
    results_df.reset_index(), root_spans, on="context.trace_id", how="inner"
).set_index("context.span_id", drop=False)

# 整理为 Phoenix 日志格式
annotation_df = to_annotation_dataframe(dataframe=results_with_spans)

await px_client.spans.log_span_annotations_dataframe(
    dataframe=annotation_df,
    annotator_kind="LLM",
)

三个追踪级评估——`relevance`、`decision path` 和 `reasoning`——被记录到 Phoenix 中真实的 Agent 追踪上，每项都有裁判给出的标签、分数和解释。这里三项都通过；`decision path` 的解释确认 Agent 按正确顺序先选择、再评论、最后总结。上面的受控案例则展示中间检查失败时的情况。

![结果](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/trace_level_all_correct.png)

## 要点

我们在相同追踪上运行了三个评估器，每个读取不同信号、回答不同问题：

- **relevance**（端点）：*最终答案看起来是否正确？*只读取 `input` 和 `output`，与端到端评估一致。
- **decision path**（中间过程）：*Agent 是否按正确顺序、使用合理参数选择正确工具？*读取 `tool_path`。
- **reasoning / support**（中间过程）：*最终答案是否真正以工具返回内容为依据？*读取 `tool_io`。

受控案例说明了三者的必要性：每个答案都看似相关，但决策路径检查发现了工具顺序错误，依据检查发现了编造的评分。这是两种*不同*失败，都无法从端点看到。核心结论是：**好看的答案可能掩盖有问题的过程，只有读取从 span 重建信号的中间过程评估才能发现。**

这一模式可以推广。对任何你关心的中间步骤，从 span 中重建相关信号并存入一列（如 `tool_path`、`tool_io`、检索上下文或子 Agent 交接），编写读取该列的裁判，与端点评估一起运行。